## 개요

| 섹션 | 주제 | 핵심 질문 |
|---|---|---|
| Section 1 | Vector RAG 기본기 | 문서를 청크로 쪼개고 임베딩해서 Neo4j에 넣고 유사도 검색 |
| Section 2 | 고급 청킹 (Parent–Child) & Stepback | 제목 기준 섹션 → Parent → Child 계층 청킹, Stepback 질문으로 검색 품질 올리기 |
| Section 3 | Text2Cypher | 자연어 → Cypher 쿼리로 그래프 질의 |
| Section 4 | Tool / Agent Routing | 벡터검색, Cypher 중 적절한 것을 선택하여 수행 |

## Prerequisites

1. `pip install -r requirements.txt` 로 주피터 노트북에 필요한 라이브러리를 설치해주세요.
1. `.env` 파일에 `API_BASE` / `API_KEY` 가 있어야 합니다. OpenRouter 또는 OpenAI API 키를 넣어주세요.
    - `API_BASE` 예시: `https://openrouter.ai/api/v1`, `https://api.openai.com/v1` 등
    - `API_KEY` 예시: `sk-xxx` 등
1. Neo4j를 사용하기 위해 Docker가 설치되어 있어야 합니다. 윈도우나 맥 사용자는 Docker Desktop을 설치하시면 괜찮습니다.
    - Docker 설치: https://docs.docker.com/get-docker/
1. Docker가 설치된 후, `docker compose up -d` 명령어로 Neo4j DB를 실행해주세요.

In [ ]:
%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2

from utils import chat, embed, neo4j_driver, chunk_text
print('Neo4j 연결:', neo4j_driver.verify_connectivity() is None)

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Neo4j 연결: True


[Neo4j Browser(http://localhost:7474)](http://localhost:7474) 에 접속해봅시다.
Neo4j Browser는 Neo4j 데이터베이스와 상호작용할 수 있는 웹 인터페이스입니다. 쿼리를 실행하고, 데이터를 시각화하며, 데이터베이스 구조를 탐색할 수 있습니다.

- Connection URL: `bolt://localhost:7687`
    - 볼트 프로토콜을 사용하여 Neo4j 데이터베이스에 연결합니다.
- Authentication type: `Username / Password`
- Username: `neo4j`
- Password: `password`


아래와 같은 화면이 나오면 성공입니다.

![](./images/000_neo4j.png){width=600px}
![](./images/001_neo4j.png){width=600px}

🚀 모든 준비가 끝났습니다! 본격적으로 실습을 진행해볼까요? 

---
## Section 1 — Vector RAG 의 출발점

**아이디어**: PDF → 텍스트 추출 → 일정 길이 청크 → 임베딩 → Neo4j `Chunk` 노드에 저장 → 질문 임베딩으로 유사도 검색.

여기서는 PDF 다운로드 대신 짧은 문자열로 동일한 파이프라인을 시연합니다.

In [166]:
sample_text = (
    "Knowledge graphs represent information as nodes and relationships. "
    "RAG combines retrieval with large language models. "
    "Neo4j is a popular graph database that supports vector indexes. "
    "Combining vector search with graph traversal is called Knowledge Graph RAG."
)

chunks = chunk_text(sample_text, 80, 10)
embeddings = embed(chunks)
print(f'청크 개수: {len(chunks)}, 임베딩 차원: {len(embeddings[0])}')

print(chunks[0])
print(embeddings[0][:3])

청크 개수: 3, 임베딩 차원: 1536
Knowledge graphs represent information as nodes and relationships. RAG combines retrieval
[-0.014447440393269062, 0.013546922244131565, -0.01237233355641365]


In [167]:
# 벡터 인덱스 생성 + 청크 적재 (이미 있으면 MERGE)
neo4j_driver.execute_query(
    """CREATE VECTOR INDEX vindex IF NOT EXISTS
       FOR (c:DemoChunk) ON c.embedding
       OPTIONS {indexConfig: {`vector.dimensions`: 1536, `vector.similarity_function`: 'cosine'}}"""
)

neo4j_driver.execute_query(
    """
    WITH $chunks AS chunks, $embeddings AS embs, range(0, size($chunks)-1) AS idx
    UNWIND idx AS i
    MERGE (c:DemoChunk {index: i})
    SET c.text = chunks[i]
    WITH c, embs[i] AS e
    CALL db.create.setNodeVectorProperty(c, 'embedding', e)
    """,
    chunks=chunks, embeddings=embeddings,
)
print(f'{len(chunks)}개의 청크 적재 완료')

3개의 청크 적재 완료


`MATCH (n) RETURN n LIMIT 25`는 Neo4j에서 처음 첫개 노드를 보여주는 쿼리입니다.
3개의 청크가 노드로 저장된 것이 확인되시면 성공입니다.

쿼리문 대신에 1 → 2 → 3 번 버튼을 차례로 눌러보셔도 괜찮습니다.

![](./images/100_3chunks.png){width=600px}

이제 직접 질문을 해봅시다. `What is Knowledge Graph RAGing?` 라는 질문에 대해 벡터 검색이 어떻게 작동하는지 살펴봅시다.
3개의 청크 중에서 가장 유사한 청크 상위 2개를 검색해봅시다.

In [168]:
question = 'What is Knowledge Graph RAGing?'
q_emb = embed([question])[0]
records, _, _ = neo4j_driver.execute_query(
    """CALL db.index.vector.queryNodes('vindex', 2, $e)
       YIELD node, score RETURN node.text AS text, score""",
    e=q_emb,
)
for r in records:
    print(round(r['score'], 3), '|', r['text'])

0.867 | Knowledge graphs represent information as nodes and relationships. RAG combines retrieval
0.816 | supports vector indexes. Combining vector search with graph traversal is called Knowledge Graph RAG.


Section 1 데모용으로 만든 `DemoChunk` 노드 3개와 벡터 인덱스를 정리합니다.

Neo4j Browser 에서 아래 Cypher 쿼리를 직접 실행해 노드와 인덱스를 정리해봅시다.

```cypher
MATCH (c:DemoChunk) DETACH DELETE c;
DROP INDEX vindex IF EXISTS;
```
`MATCH (n) RETURN n LIMIT 25` 로 다시 조회했을 때 아무 노드도 나오지 않으면 성공입니다.

In [169]:
neo4j_driver.execute_query(
    """MATCH (c:DemoChunk) DETACH DELETE c""",
)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x31067a9c0>, keys=[])

---
## Section 2 — 더 좋은 청킹과 Stepback 프롬프팅

**아이디어**: 문서를 정적 길이로 자르지 말고 **제목 기준**으로 자르고, 사용자의 질문은 일반화한 "Stepback 질문"으로 다시 써서 검색 적중률을 올려봅시다.

이번 섹션에서는 pdf를 받아서 실습을 진행해보겠습니다.
먼저 pdf파일에서 텍스트를 추출해봅시다.

In [170]:
import requests

remote_pdf_url = "https://arxiv.org/pdf/1709.00666.pdf"
pdf_filename = "downloaded.pdf"

response = requests.get(remote_pdf_url)

if response.status_code == 200:
    with open(pdf_filename, "wb") as pdf_file:
        pdf_file.write(response.content)
else:
    print("Failed to download the PDF. Status code:", response.status_code)

`downloaded.pdf` 라는 파일이 현재 작업 디렉토리에 생겼을 것입니다. PDF에서 텍스트가 잘 추출되면 성공입니다.

In [171]:
import pdfplumber

pdf_text = ""

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        pdf_text += page.extract_text()

print(pdf_text[0:500])

Einstein’s Patents and Inventions
Asis Kumar Chaudhuri
Variable Energy Cyclotron Centre
1‐AF Bidhan Nagar, Kolkata‐700 064
Abstract: Times magazine selected Albert Einstein, the German born Jewish Scientist as the person of the 20th
century. Undoubtedly, 20th century was the age of science and Einstein’s contributions in unravelling mysteries
of nature was unparalleled. However, few are aware that Einstein was also a great inventor. He and his
collaborators had patented a wide variety of inventi


제목 기반으로 청킹을 해봅시다. `split_text_by_titles` 함수를 이용해서 PDF 텍스트를 제목 기준으로 나눠봅시다.
Regex로 제목을 인식해서 텍스트를 나눠봅시다.

In [172]:
import re

def split_text_by_titles(text):
    # 줄 시작이 숫자(+선택적 대문자) + ". " + 짧은 제목 인 라인을 제목으로 간주
    title_pattern = re.compile(r"(\n\d+[A-Z]?\. {1,3}.{0,60}\n)", re.DOTALL)
    titles = title_pattern.findall(text)
    sections = re.split(title_pattern, text)

    sections_with_titles = [sections[0]]
    for i in range(1, len(titles) + 1):
        section_text = sections[i * 2 - 1].strip() + "\n" + sections[i * 2].strip()
        sections_with_titles.append(section_text)
    return sections_with_titles


pdf_sections = split_text_by_titles(pdf_text)
print(f"Number of sections: {len(pdf_sections)}")

pdf_embeddings = embed(pdf_sections)
print(f"PDF 임베딩 개수: {len(pdf_embeddings)}, 차원: {len(pdf_embeddings[0])}")

Number of sections: 9
PDF 임베딩 개수: 9, 차원: 1536


In [173]:
for c in pdf_sections:
    print(c[:100])
    print('---')

Einstein’s Patents and Inventions
Asis Kumar Chaudhuri
Variable Energy Cyclotron Centre
1‐AF Bidhan 
---
1. Introduction
Towards the end of the last century, Times Magazine asked some of the World’s leadin
---
2. Einstein’s life in brief
Albert Einstein was born in a Jewish family on March 14, 1879 in a small
---
63. Einstein’s Inventions and Patents
Table 1: Patents of Jacob Einstein (Albert Einstein's uncle).

---
3A. Einstein‐Szilard Refrigeration system
Among all of Einstein's inventions, possibly the most impo
---
113B. Sound reproduction system with Rudolf Goldschmidt
Rudolf Goldschmidt was a German Engineer and
---
3C. Light Intensity self‐adjusting Camera with Gustav Bucky
Einstein, with his long‐time friend Gust
---
3D. Einstein's Design of a Blouse:
It is rather amusing to note that Einstein was interested in clot
---
4. Conclusions
Einstein’s inventions and patents are mostly of historical importance now. His invent
---


제목 기준으로 나눈 청크들을 확인해봅시다. 아직은 덩어리가 너무 크기 때문에 소분해서 chunking을 해봅시다.

```{mermaid}
graph LR
    A[PDF] -- has parent --> B1[Section, Parent Chunk, Vector indexed]
    A[PDF] -- has parent --> B2[Section, Parent Chunk, Vector indexed]
    B1 -- has child --> C1[Child Chunk, Vector indexed]
    B1 -- has child --> C2[Child Chunk, Vector indexed]
    B2 -- has child --> C3[Child Chunk, Vector indexed]
    B2 -- has child --> C4[Child Chunk, Vector indexed]
```

In [174]:
parent_chunks = []
for i, s in enumerate(pdf_sections, start=1):
    print(f"{f'Section {i}':-^50}")
    _chunks = chunk_text(s, 2000, 40)
    parent_chunks.extend(_chunks)
    for c in _chunks:
        print(c[:100])
        print('---')

--------------------Section 1---------------------
Einstein’s Patents and Inventions
Asis Kumar Chaudhuri
Variable Energy Cyclotron Centre
1‐AF Bidhan 
---
--------------------Section 2---------------------
1. Introduction
Towards the end of the last century, Times Magazine asked some of the World’s leadin
---
--------------------Section 3---------------------
2. Einstein’s life in brief
Albert Einstein was born in a Jewish family on March 14, 1879 in a small
---
education, Einstein remained in Munich. Pavia factory was also a
failure. Though Hermann Einstein st
---
festival running from mid or late September to the first weekend in October, with more than 6 millio
---
twenty to pay for his Swiss naturalization papers. Einstein’s allowance was modest not meager.
In 18
---
and unlike Mileva, was not a brilliant woman. She took good care of
Einstein and little by little sh
---
re‐derived diffusion equation and expressed Avogadro’s number in
experimentally determined quantitie
---
Wein (1

이제 Neo4j 에 적재해봅시다.

In [175]:
cypher_import_query = """
MERGE (pdf:PDF {id:$pdf_id})
MERGE (p:Parent {id:$pdf_id + '-' + $id})
SET p.text = $parent
MERGE (pdf)-[:HAS_PARENT]->(p)
WITH p, $children AS children, $embeddings as embeddings
UNWIND range(0, size(children) - 1) AS child_index
MERGE (c:Child {id: $pdf_id + '-' + $id + '-' + toString(child_index)})
SET c.text = children[child_index], c.embedding = embeddings[child_index]
MERGE (p)-[:HAS_CHILD]->(c);
"""

for i, chunk in enumerate(parent_chunks):
    child_chunks = chunk_text(chunk, 500, 20)
    embeddings = embed(child_chunks)
    # Add to neo4j
    neo4j_driver.execute_query(
        cypher_import_query,
        id=str(i),
        pdf_id="1709.00666",
        parent=chunk,
        children=child_chunks,
        embeddings=embeddings,
    )

아래 그림과 같이 나오면 성공입니다. PDF → Section(Parent Chunk) → Child Chunk 의 계층이 만들어졌습니다.

![](./images/101_parent_child.png){width=600px}

parent chunk를 검색하기 위한 벡터 인덱스를 만들어봅시다.

In [176]:
index_name = "parent"
neo4j_driver.execute_query(
    """
    CREATE VECTOR INDEX parent IF NOT EXISTS
    FOR (c:Child)
    ON c.embedding
    """)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x1517046d0>, keys=[])

질문과 가장 유사한 child chunk를 검색하고, 해당되는 parent chunk를 가져와봅시다. 

In [177]:
retrieval_query = """
CALL db.index.vector.queryNodes($index_name, $k * 4, $question_embedding)
YIELD node, score
MATCH (node)<-[:HAS_CHILD]-(parent)
WITH parent, max(score) AS score
RETURN parent.text AS text, score
ORDER BY score DESC
LIMIT toInteger($k)
"""

def parent_retrieval(question: str, index_name: str, k: int = 4) -> list[str]:
    question_embedding = embed([question])[0]

    similar_records, _, _ = neo4j_driver.execute_query(
        retrieval_query,
        question_embedding=question_embedding,
        k=k,
        index_name=index_name,
    )

    return [record["text"] for record in similar_records]

documents = parent_retrieval(
    "Who was the Einsten's collaborator on sound reproduction system?",
    "parent",
    k=4
)

for doc in documents:
    print(doc[:200])
    print('---')

113B. Sound reproduction system with Rudolf Goldschmidt
Rudolf Goldschmidt was a German Engineer and inventor. He earned his engineering degree
in 1898 and PhD in 1906. He spent a decade working in En
---
entitled,
“Device, especially for sound‐reproduction equipment, in which changes of an electric current
generate movements of a magnetised body by means of magnetostriction.” As the title suggests, th
---
used to work upon on solving mathematical problems (not
related to his ongoing theoretical investigations) or took upon some practical problem. As shown in
Table. 2, Einstein was involved in three maj
---
Wein (1864‐1928) was a German physicist who did pioneering work on radiation. He
discovered the displacement law (the wavelength changes with temperature). He also introduced the
concept of ideal or b
---


히트율을 높이기 위해 stepback 프롬프트를 만들어봅시다.

In [178]:
stepback_system_message = """
You are an expert at world knowledge. Your task is to step back
and paraphrase a question to a more generic step-back question, which
is easier to answer. Here are a few examples

"input": "Could the members of The Police perform lawful arrests?"
"output": "what can the members of The Police do?"

"input": "Jan Sindel’s was born in what country?"
"output": "what is Jan Sindel’s personal history?"
"""


def generate_stepback(question: str):
    user_message = f"""{question}"""
    step_back_question = chat(
        messages=[
            {"role": "system", "content": stepback_system_message},
            {"role": "user", "content": user_message},
        ]
    )
    return step_back_question


original_question = "Who was the Einsten's collaborator on sound reproduction system?"
stepped_back_question = generate_stepback(original_question)

print("Original question:", original_question)
print("Stepped-back question:", stepped_back_question)

Original question: Who was the Einsten's collaborator on sound reproduction system?
Stepped-back question: Who worked with Einstein on technological projects?


In [179]:


def generate_answer(question: str, documents: list[str]) -> str:
    answer_system_message = "You're en Einstein expert, but can only use the provided documents to respond to the questions."
    user_message = f"""
    Use the following documents to answer the question that will follow:
    {documents}

    ---

    The question to answer using information only from the above documents: {question}
    """
    result = chat(
        messages=[
            {"role": "system", "content": answer_system_message},
            {"role": "user", "content": user_message},
        ]
    )
    print("Response:", result)


def rag_pipeline(question: str) -> str:
    stepback_prompt = generate_stepback(question)
    print(f"Stepback prompt: {stepback_prompt}")
    documents = parent_retrieval(stepback_prompt, index_name="parent", k=4)
    answer = generate_answer(question, documents)
    return answer


rag_pipeline("Who was the Einsten's collaborator on sound reproduction system?")

Stepback prompt: Who worked with Einstein on technological projects?
Response: Einstein's collaborator on the sound reproduction system was Rudolf Goldschmidt.


---
## Section 3 — Text2Cypher

**아이디어**: 그래프 스키마 + few-shot 예시를 LLM 프롬프트에 넣어 자연어를 Cypher 로 변환한다.  

In [180]:
from schema_utils import get_schema

prompt_template = """
Instructions: 
Generate Cypher statement to query a graph database to get the data to answer the user question below.

Graph Database Schema:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided in the schema.
{schema}

Terminology mapping:
This section is helpful to map terminology between the user question and the graph database schema.
{terminology}

Examples:
The following examples provide useful patterns for querying the graph database.
{examples}

Format instructions:
Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to 
construct a Cypher statement.
Do not include any text except the generated Cypher statement.
ONLY RESPOND WITH CYPHER, NO CODEBLOCKS.

User question: {question}
"""

schema_string = get_schema(neo4j_driver)

terminology_string = """
PDF: When a user asks about a PDF by trade like document, paper, they are referring to a node with the label 'PDF'.
Parent: When a user asks about a section, chapter, part, they are referring to a node with the label 'Parent'.
Child: When a user asks about a subsection, detail, they are referring to a node with the label 'Child'.
"""

examples_string = [
    (
        "Which sections (parent) are in the PDF about '1709.00666'?",
        """
        MATCH (pdf:PDF {id: '1709.00666'})-[:HAS_PARENT]->(p:Parent)
        RETURN p.text
        """,
    ),
]

def generate_cypher(question: str) -> str:
    cypher = prompt_template.format(
        schema=schema_string,
        terminology=terminology_string,
        examples="\n\n".join([f"Question: {q}\nCypher: {c}" for q, c in examples_string]),
        question=question,
    )
    return cypher

question = "How many sections does the PDF '1709.00666' have?"
cypher_prompt = generate_cypher(question)
print(cypher_prompt)


Instructions: 
Generate Cypher statement to query a graph database to get the data to answer the user question below.

Graph Database Schema:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided in the schema.
Node properties:
Chunk {index: INTEGER, text: STRING, embedding: LIST}
PDF {id: STRING}
Parent {text: STRING, id: STRING}
Child {text: STRING, id: STRING, embedding: LIST}
Relationship properties:

The relationships:
(:PDF)-[:HAS_PARENT]->(:Parent)
(:Parent)-[:HAS_CHILD]->(:Child)

Terminology mapping:
This section is helpful to map terminology between the user question and the graph database schema.

PDF: When a user asks about a PDF by trade like document, paper, they are referring to a node with the label 'PDF'.
Parent: When a user asks about a section, chapter, part, they are referring to a node with the label 'Parent'.
Child: When a user asks about a subsection, detail, they are re

In [181]:
cypher = chat(messages=[{"role": "user", "content": cypher_prompt}])
print(cypher)

MATCH (pdf:PDF {id: '1709.00666'})-[:HAS_PARENT]->(p:Parent) RETURN count(p)


In [182]:
from text2cypher import Text2Cypher
t2c = Text2Cypher(driver=neo4j_driver)

t2c.set_prompt_section("question", question)
t2c.set_prompt_section("terminology", terminology_string)
t2c.set_prompt_section("examples", examples_string)

cypher_from_class = t2c.generate_cypher()

print(cypher_from_class)

MATCH (pdf:PDF {id: '1709.00666'})-[:HAS_PARENT]->(p:Parent) RETURN count(p) AS section_count


In [183]:
records, *_ = neo4j_driver.execute_query(
    cypher_from_class
)

print(f"Number of parents: {records[0]['section_count']}")

Number of parents: 27


실제로 Neo4j 에 적재된 그래프와 동일한 것을 볼 수 있습니다.

![](./images/300_text2cypher.png){width=600px}

---
## Section 4 — 도구 라우팅

**아이디어**: "벡터검색", "Cypher 검색" 등 여러 도구를 LLM 에게 보여주고, 질문에 맞는 도구를 **LLM 이 직접 고르게** 한다. 그리고 답변이 부족하면 다시 도구를 호출한다(self-critique 루프).

여기서는 **단일 도구 선택 호출**만 시연합니다.

In [184]:
from utils import tool_choice

tools = [
    {'type': 'function', 'function': {
        'name': 'vector_search',
        'description': 'Search unstructured text chunks by semantic similarity.',
        'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']},
    }},
    {'type': 'function', 'function': {
        'name': 'cypher_search',
        'description': 'Run a Cypher query over the movies knowledge graph.',
        'parameters': {'type': 'object', 'properties': {'question': {'type': 'string'}}, 'required': ['question']},
    }},
]

calls = tool_choice(
    [
        {'role': 'user', 'content': 'Who directed The Matrix?'},
    ],
    tools=tools,
)


questions = ["What year was the matrix released?", "How does matrix multiplication work?"]


for q in questions:
    calls = tool_choice(
        [
            {'role': 'user', 'content': q},
        ],
        tools=tools,
    )
    print(q)
    print(calls[0].function.name)
    print('---')

What year was the matrix released?
cypher_search
---
How does matrix multiplication work?
vector_search
---
